In [1]:
import requests

# Get top 100 Binance symbols by 24h quote volume

url = "https://api.binance.com/api/v3/ticker/24hr"
resp = requests.get(url, timeout=10)
resp.raise_for_status()
tickers = resp.json()  # list of dicts, each has 'symbol' and 'quoteVolume'

# sort by quoteVolume (fallback to 0) and take top 100 symbols
top100_symbols = [
    t["symbol"]
    for t in sorted(
        tickers,
        key=lambda x: float(x.get("quoteVolume", 0) or 0),
        reverse=True,
    )[:100]
]

print(top100_symbols)

['USDTIDRT', 'BUSDBIDR', 'BTCBIDR', 'ETHBIDR', 'ALICEBIDR', 'USDTBIDR', 'SANDBIDR', 'FTMBIDR', 'DOTBIDR', 'XRPBIDR', 'SLPBIDR', 'AVAXBIDR', 'ADABIDR', 'DOGEBIDR', 'SOLBIDR', 'MANABIDR', 'BNBBIDR', 'MATICBIDR', 'TNSRTRY', 'BTCJPY', 'TKOBIDR', 'ZILBIDR', 'BTCUSDT', 'USDTTRY', 'XRPJPY', 'NBTBIDR', 'AXSBIDR', 'ETHJPY', 'BTCFDUSD', 'ETHUSDT', 'USDCUSDT', 'BTCNGN', 'USDTARS', 'USDTCOP', 'DYMTRY', 'XRPTRY', 'ETHFDUSD', 'USDCTRY', 'USDTNGN', 'BTCTRY', 'PARTITRY', 'BTCUSDC', 'BUSDRUB', 'ALICETRY', 'ZECUSDT', 'ETHTRY', 'MAVTRY', 'SOLUSDT', 'ADAJPY', 'GIGGLETRY', 'MMTTRY', 'XRPFDUSD', 'LAYERTRY', 'FDUSDUSDT', 'XRPUSDT', 'ASTERTRY', 'ETHUSDC', 'SOLFDUSD', 'STRKTRY', 'BNBJPY', 'XAITRY', 'PEPETRY', 'ASTERUSDT', 'HIGHTRY', '0GTRY', 'NILTRY', 'ALLOTRY', 'SOLTRY', 'TNSRUSDT', 'GPSTRY', 'RESOLVTRY', 'BNBUSDT', 'HBARTRY', 'TRXTRY', 'NTRNTRY', 'TLMTRY', 'SOLJPY', 'BNBFDUSD', 'SUIJPY', 'USDTBRL', 'DOGEUSDT', 'EIGENTRY', 'XVGTRY', 'BNBUSDC', 'BUSDUAH', 'FETTRY', 'TSTTRY', 'STRAXTRY', 'ENATRY', 'DOGEFDUSD', 

In [ ]:
import os
import time
import pandas as pd
from pathlib import Path
from requests.exceptions import HTTPError

# settings
SYMBOL_LIST = top100_symbols  # uses variable already defined in notebook
INTERVAL = "15m"
INTERVAL_MS = 15 * 60 * 1000
LIMIT = 1000  # Binance max klines per request
MAX_WINDOW_MS = LIMIT * INTERVAL_MS
KLINES_ENDPOINT = "https://api.binance.com/api/v3/klines"
OUT_DIR = Path("data")
OUT_DIR.mkdir(exist_ok=True)

def _request_klines(symbol: str, start_ms: int, end_ms: int, retries=3, backoff=1.0):
    params = {
        "symbol": symbol,
        "interval": INTERVAL,
        "startTime": int(start_ms),
        "endTime": int(end_ms),
        "limit": LIMIT,
    }
    for attempt in range(retries):
        try:
            r = requests.get(KLINES_ENDPOINT, params=params, timeout=10)
            r.raise_for_status()
            return r.json()
        except HTTPError as e:
            # on rate limit or server error, back off and retry
            if attempt + 1 == retries:
                raise
            time.sleep(backoff * (attempt + 1))
        except Exception:
            if attempt + 1 == retries:
                raise
            time.sleep(backoff * (attempt + 1))
    return []

for symbol in SYMBOL_LIST:
    all_klines = []
    while True:
        end_ms = MAX_WINDOW_MS - 1
        try:
            klines = _request_klines(symbol, None, end_ms)
        except Exception as e:
            print(f"Failed to fetch {symbol} klines: {e}")
            break

        if not klines:
            # no data returned for this window -> stop
            break

        all_klines.extend(klines)

        last_open = klines[-1][0]  # open time of last returned candle
        # progress check: if no progress, break to avoid infinite loop
        if last_open + INTERVAL_MS >= end_ms:
            break

        # move to next candle after the last returned
        start_ms = last_open + INTERVAL_MS

        # polite pause to avoid hitting rate limits
        time.sleep(0.2)

    if not all_klines:
        print(f"No klines collected for {symbol}, skipping.")
        continue

    # create DataFrame and keep relevant columns
    df = pd.DataFrame(
        all_klines,
        columns=[
            "open_time",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "close_time",
            "quote_asset_volume",
            "number_of_trades",
            "taker_buy_base_asset_volume",
            "taker_buy_quote_asset_volume",
            "ignore",
        ],
    )
    df = df[["open_time", "open", "high", "low", "close", "volume"]].copy()
    df = df.rename(columns={"open_time": "timestamp"})
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.sort_values("timestamp").reset_index(drop=True)

    out_path = OUT_DIR / f"{symbol}_15m.csv"
    df.to_csv(out_path, index=False)
    print(f"Saved {len(df)} rows for {symbol} -> {out_path}")

No klines collected for USDTIDRT, skipping.
No klines collected for BUSDBIDR, skipping.
No klines collected for BTCBIDR, skipping.
No klines collected for ETHBIDR, skipping.
No klines collected for ALICEBIDR, skipping.
No klines collected for USDTBIDR, skipping.
No klines collected for SANDBIDR, skipping.
No klines collected for FTMBIDR, skipping.
No klines collected for DOTBIDR, skipping.
No klines collected for XRPBIDR, skipping.
No klines collected for SLPBIDR, skipping.
No klines collected for AVAXBIDR, skipping.
No klines collected for ADABIDR, skipping.
No klines collected for DOGEBIDR, skipping.
No klines collected for SOLBIDR, skipping.
No klines collected for MANABIDR, skipping.
No klines collected for BNBBIDR, skipping.
No klines collected for MATICBIDR, skipping.
No klines collected for TNSRTRY, skipping.
No klines collected for BTCJPY, skipping.
No klines collected for TKOBIDR, skipping.
No klines collected for ZILBIDR, skipping.
No klines collected for BTCUSDT, skipping.
N